# Topic Discovery and Document Recommendation Using NMF

When faced with thousands of documents, manually identifying topics and finding similar content is impossible for a human. NMF (Non-negative Matrix Factorization) solves this by decomposing documents into a mix of hidden topics. Since NMF only works with non-negative values, we first convert text to numbers using TF-IDF, which scores how important each word is to each document. NMF then discovers the underlying topics and expresses each document as a combination of those topics — enabling both topic analysis and content recommendation.

**Author:** Mohamed Chaker Iben Hadj Amor  
**Dataset:** 20 Newsgroups (18,000+ posts across 20 categories)  
**Tools:** Python, scikit-learn (NMF, TF-IDF, t-SNE), matplotlib

## 1. Load Data

In [1]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all')
print(f"Total documents: {len(data.data)}")
print(f"Categories: {len(data.target_names)}") 

Total documents: 18846
Categories: 20


## 2. TF-IDF and NMF Topic Discovery

We convert text to numbers using TF-IDF, filtering out common English words and email header junk (edu, com, nntp, etc.) that appear in every document but carry no meaning. We keep the top 10,000 most important words.

NMF then decomposes the TF-IDF matrix into 5 topics. Each topic is defined by the words that score highest in that component.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from sklearn.preprocessing import Normalizer
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Remove common English words + email header junk that causes noise
custom_stop = ENGLISH_STOP_WORDS.union([
    'edu', 'com', 'subject', 'lines', 'organization', 'writes', 'article',
    'posting', 'nntp', 'host', 'university', 'netcom', 'from', 're', 'cs',
    'ac', 'hp', 'sun', 'ibm', 'ca', 'stratus'
])

# Convert text to TF-IDF features (top 10,000 words, junk removed)
tfidf = TfidfVectorizer(max_features=10000, stop_words=list(custom_stop))
articles = tfidf.fit_transform(data.data)

# Discover 5 topics using NMF
nmf = NMF(n_components=5, max_iter=1000)
X_transformed = nmf.fit_transform(articles)

# Display the top 10 words for each topic
words = tfidf.get_feature_names_out()
for i, component in enumerate(nmf.components_):
    top_words_idx = component.argsort()[-10:][::-1]
    top_words = [words[j] for j in top_words_idx]
    print(f"Topic {i}: {', '.join(top_words)}")

Topic 0: game, team, games, hockey, year, players, baseball, cmu, play, season
Topic 1: windows, drive, dos, thanks, card, file, scsi, pc, use, software
Topic 2: god, jesus, bible, christian, christians, christ, faith, believe, sin, people
Topic 3: key, clipper, chip, encryption, keys, access, escrow, government, digex, algorithm
Topic 4: people, don, just, like, think, state, right, car, gun, government


### Topics Discovered

| Topic | Top Words | Interpretation |
|-------|-----------|---------------|
| 0 | game, team, hockey, baseball, players | **Sports** |
| 1 | windows, drive, dos, card, scsi | **Computer Hardware** |
| 2 | god, jesus, bible, christian, faith | **Religion** |
| 3 | key, clipper, chip, encryption, escrow | **Encryption / Security** |
| 4 | people, gun, government, israel, state | **Politics / Guns** |

NMF discovered 5 meaningful topics from raw text with zero labels.

## 3. Document Recommender

Using cosine similarity on the NMF features, we can find documents with similar topic mixtures. We normalize each document's topic vector to unit length, then compute the dot product — this gives cosine similarity (1.0 = identical topic mix, 0.0 = completely different).

In [3]:
# Normalize topic features for cosine similarity
normalizer = Normalizer()
X_norm = normalizer.fit_transform(X_transformed)

# Pick a document and find the 5 most similar
current_doc = X_norm[500, :]
similarities = X_norm.dot(current_doc)

top_idx = similarities.argsort()[::-1][1:6]
print("Documents most similar to document 500:\n")
for idx in top_idx:
    print(f"Similarity: {similarities[idx]:.3f} | {data.data[idx][:130]}")
    print()

Documents most similar to document 500:

Similarity: 0.992 | From: reedr@cgsvax.claremont.edu
Subject: Re: DID HE REALLY RISE???
Organization: The Claremont Graduate School
Lines: 9

In artic

Similarity: 0.991 | From: koberg@spot.Colorado.EDU (Allen Koberg)
Subject: Re: _Christianity In Crisis_ by Hank Hanegraaff
Organization: University of

Similarity: 0.990 | From: todd@nickel.laurentian.ca
Subject: Re: Homosexuality issues in Christiani
Organization: Laurentian University
Lines: 13

whi

Similarity: 0.989 | From: JEK@cu.nih.gov
Subject: Trinity
Lines: 27

James Green writes:

 > Can't someone describe someone's trinity in simple declar

Similarity: 0.986 | From: healta@saturn.wwc.edu (Tammy R Healy)
Subject: Re: YOU WILL ALL GO TO HELL!!!
Lines: 31
Organization: Walla Walla College
Di



## 4. t-SNE Visualization

We assign each document its dominant topic (the topic with the highest NMF weight) and use t-SNE to project all documents into 2D. Colors represent the dominant topic.

In [ ]:
# Assign each document its dominant topic
nmf_labels = np.argmax(X_transformed, axis=1)

# Reduce to 2D with t-SNE
tsne = TSNE(learning_rate=200, random_state=42)
X_2d = tsne.fit_transform(X_transformed)

# Plot colored by dominant topic
plt.figure(figsize=(10, 7))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=nmf_labels, cmap='viridis', s=5)
plt.colorbar(label='Dominant Topic')
plt.title('t-SNE Visualization Colored by NMF Topic')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.show()

![t-SNE Visualization](tsne_plot.png)

## 5. Conclusion

NMF successfully discovered 5 distinct topics from 18,000+ newsgroup documents with zero labels:

- **Topic 0 — Sports:** hockey, baseball, game, team, players
- **Topic 1 — Computer Hardware:** windows, drive, dos, scsi, card
- **Topic 2 — Religion:** god, jesus, bible, christian, faith
- **Topic 3 — Encryption/Security:** key, clipper, chip, encryption, escrow
- **Topic 4 — Politics/Guns:** people, gun, government, israel, state

The t-SNE visualization confirms clear separation between topics with minimal overlap. The recommender system correctly identifies similar documents — when given a computer-related post, it recommends other computer posts.

NMF was the right choice here because TF-IDF values are non-negative, and the results are interpretable — each topic can be understood by looking at its top words. This approach can be applied to any text collection: news articles, customer reviews, research papers, or support tickets.